# Multispectral Sensor Characterization Using SPION Concentration Series — Data Analysis

## Overview

This notebook documents the analysis of the responses of three physical multispectral sensors to SPION suspensions at different iron concentrations.

The sensors were measured sequentially in three separate measurement series. Experimental steps were recorded in software-generated marker CSV files. Sensor signals and markers are aligned separately for each measurement series.

The analysis determines the mean ADC signal within each calibration window and summarizes three repeated measurements per concentration. Sensor responses are compared using the mean and sample standard deviation of the replicate means.

The stock suspension has an iron concentration of 5 mg Fe/mL, corresponding to a relative concentration of 100%. Model fitting is outside the scope of this initial analysis.

Differences between measurement series may reflect both sensor characteristics and differences in measurement conditions.

## Experimental Metadata

### Measurement Conditions

- **Experiment:** SPION concentration series
- **Operator:** Lukas Kroll
- **Location:** Laboratory A701, FH Dortmund
- **Measurement date:** 21 April 2026
- **Acquisition:** Three sequential static measurement series, one per physical sensor
- **Batch identifier:** XL004
- **Stock iron concentration:** 5 mg Fe/mL
- **Relative concentrations:** 0, 2.5, 5, 10, 25, 50, 75, and 100%
- **Warm-up time:** 20 min
- **min. sample time:** 60 s
- **Replicates:** Three measurements per concentration

Relative concentration is expressed as a percentage of the stock concentration. The corresponding iron concentration is calculated as:

$$
c_{\mathrm{Fe}} = \frac{c_{\mathrm{rel}}}{100}\,c_{\mathrm{stock}}, \qquad c_{\mathrm{stock}} = 5\,\mathrm{mg\,Fe\,mL^{-1}}.
$$

### Measurement Series

The physical sensor identity is assigned through the measurement-series documentation. The software ID recorded in the sensor CSV files is 1000 for all three series and therefore does not distinguish the physical devices.

| Physical sensor | Recorded software ID | Acquisition start, as recorded in the protocol |
|---|---|---|
| Sensor 0 | 1000 | 2026-04-21 10:38:26.888558 |
| Sensor 1 | 1000 | 2026-04-21 12:48:24.226562 |
| Sensor 2 | 1000 | 2026-04-21 14:23:54.114110 |

The protocol timestamps do not specify a time zone.

### Sensor Configuration

Acquisition settings were taken from the `Protocol_*.txt` file accompanying each measurement series. All three protocols report identical settings.

- **Device class:** SpectralSensor
- **Timeline band:** Band 9 (640 nm)
- **ATIME:** 0
- **ASTEP:** 8993
- **GAIN:** 1
- **IRMAIN:** 25
- **IROPP:** 25
- **VISM:** 255
- **VISLR:** 255
- **Signal unit:** ADC counts
- **Source timestamp unit:** Milliseconds

These settings are reported as recorded by the acquisition software. The interpretation of `GAIN` as a physical amplification factor and the conversion of `ATIME` and `ASTEP` to integration time require confirmation from the device configuration or driver documentation.

### Data Sources

Each measurement directory contains a sensor CSV file, a marker CSV file, and a protocol TXT file.

| Physical sensor | Measurement directory |
|---|---|
| Sensor 0 | `Sensor0_Messung_20260421_103826` |
| Sensor 1 | `Sensor1_Messung_20260421_124824` |
| Sensor 2 | `Sensor2_Messung_20260421_142354` |

- **Base directory:** `experiments/E01_sensor_calibration/data/`
- **Sensor measurements:** `SpectralSensor 1_<timestamp>.csv`
- **Experimental events:** `Markers_<timestamp>.csv`
- **Acquisition metadata:** `Protocol_<timestamp>.txt`

The sensor CSV filename is not used to infer physical sensor identity. Source files remain unchanged throughout the analysis.

## Analysis Parameters

- **Analyzed wavelength:** 640 nm
- **Calibration window:** 57 s after each selected marker
- **Signal statistic:** Arithmetic mean of valid ADC values within each window
- **Replicate summary:** Mean and sample standard deviation of three window means per concentration and sensor
- **Replicate weighting:** Equal weight for each window mean
- **Signal processing:** No baseline correction or reference normalization
- **Error bars:** Sample standard deviation between replicate means, not the standard deviation of individual samples within a window

### Figure Style

Colours and markers identify physical sensors consistently across figures, independently of recorded software IDs or sensor positions.

| Physical sensor | Colour | Hex code | Marker |
|---|---|---|---|
| Sensor 0 | Dark blue | `#004488` | Circle |
| Sensor 1 | Ochre | `#DDAA33` | Square |
| Sensor 2 | Rose | `#BB5566` | Triangle |


## Analysis Workflow

1. Read the sensor measurements and marker files for each measurement series.
2. Inspect data columns, missing values, and time ranges.
3. Establish a common time origin for each sensor measurement and its associated markers.
4. Plot the raw signals and inspect the calibration markers.
5. Assign calibration markers to known concentrations and replicate numbers.
6. Select and visually verify the calibration windows.
7. Calculate the valid sample count, mean ADC signal, and sample standard deviation within each window.
8. Summarize the three replicate means at each concentration.
9. Plot sensor response against iron concentration and compare the three sensors.
10. Export the result tables, figures, and analysis settings.

## Configuration

`measurement_series` links each physical sensor to its sensor file, marker file, protocol file, recorded software ID, and acquisition timestamp. `stock_concentration_mg_fe_ml` defines the stock iron concentration. `wavelength_nm`, `window_s`, and `expected_replicates` define the analysis settings. `sensor_styles` assigns consistent colours and markers to the physical sensors.

`Path()` constructs file paths relative to the notebook location. This configuration block defines inputs and settings; it does not read or modify the measurement files.


### Import PATH and LIB


`project_folder` identifies the repository root. `data_folder` contains the three calibration measurement series, and `output_folder` defines the destination for derived results. Adding the project folder to `sys.path` enables imports from `src.spectral_sensor`.

`read_sensor_csv()` and `read_marker_rows()` load measurement data. `read_protocol()` extracts the currently supported protocol metadata. `synchronize_time()` and `select_time_window()` prepare the time axes and calibration windows. `plot_timeseries()` displays the selected signals.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

# Repository and data paths
project_folder = Path("..").resolve()
data_folder = project_folder / "experiments/E01_sensor_calibration/data"
output_folder = project_folder / "experiments/E01_sensor_calibration/results"

if str(project_folder) not in sys.path:
    sys.path.insert(0, str(project_folder))

# Pipeline functions
from src.spectral_sensor.io import read_sensor_csv, read_marker_rows, read_protocol
from src.spectral_sensor.preprocessing import synchronize_time, marker_time, select_time_window
from src.spectral_sensor.plotting import plot_timeseries

### Load Data

In [4]:
# Experimental metadata
experiment_id = "E01_sensor_calibration"
operator = "Lukas Kroll"
location = "Laboratory A701, FH Dortmund"

measurement_series = {
    "sensor_0": {
        "physical_sensor_id": 0,
        "recorded_sensor_id": 1000,
        "folder": data_folder / "Sensor0_Messung_20260421_103826",
        "file_timestamp": "20260421_103826",
    },
    "sensor_1": {
        "physical_sensor_id": 1,
        "recorded_sensor_id": 1000,
        "folder": data_folder / "Sensor1_Messung_20260421_124824",
        "file_timestamp": "20260421_124824",
    },
    "sensor_2": {
        "physical_sensor_id": 2,
        "recorded_sensor_id": 1000,
        "folder": data_folder / "Sensor2_Messung_20260421_142354",
        "file_timestamp": "20260421_142354",
    },
}

# Analysis parameters
stock_concentration_mg_fe_ml = 5.0
wavelength_nm = 640
window_s = 57.0
expected_replicates = 3

# Tol High Contrast: physical sensor identities
sensor_styles = {
    "sensor_0": {"color": "#004488", "marker": "o"},
    "sensor_1": {"color": "#DDAA33", "marker": "s"},
    "sensor_2": {"color": "#BB5566", "marker": "^"},
}

sensor_data = {}
marker_data = {}
protocol_metadata = {}

for sensor_name, measurement in measurement_series.items():
    folder = measurement["folder"]
    timestamp = measurement["file_timestamp"]

    sensor_file = folder / f"SpectralSensor 1_{timestamp}.csv"
    marker_file = folder / f"Markers_{timestamp}.csv"
    protocol_file = folder / f"Protocol_{timestamp}.txt"

    sensor_data[sensor_name] = read_sensor_csv(sensor_file)
    marker_data[sensor_name] = read_marker_rows(marker_file)
    protocol_metadata[sensor_name] = read_protocol(protocol_file)

pd.DataFrame.from_dict(protocol_metadata, orient="index")

,protocol_file,experiment_name,batch_number,start_time,timeline_band_nm,timeline_bands_nm
sensor_0,Protocol_20260421_103826.txt,Sensor 0 Calibration,XL004,2026-04-21T10:38:26.888558,640,[640]
sensor_1,Protocol_20260421_124824.txt,Sensor 1 Calibration,XL004,2026-04-21T12:48:24.226562,640,[640]
sensor_2,Protocol_20260421_142354.txt,Sensor 2 Calibration,XL004,2026-04-21T14:23:54.114110,640,[640]
